# Sistemas expertos: encadenamiento hacia delante y hacia atrás

**Facsímil 12 · Lógica, conocimiento e incertidumbre** — capítulo 4 (sistemas expertos y soporte a la decisión).

Mucho antes de los grandes modelos de lenguaje, la inteligencia artificial ya tomaba decisiones útiles
en banca, medicina o soporte técnico. Lo hacía con **sistemas expertos**: una base de reglas
*SI… ENTONCES…* escritas por personas y un **motor de inferencia** que las encadena para llegar a una
conclusión. No aprenden de datos; razonan con conocimiento explícito. Y, a diferencia de una red neuronal,
**te pueden explicar por qué** han concluido lo que han concluido, regla a regla.

En este cuaderno construyes ese motor con NumPy y Python puro, sobre un caso realista —la **aprobación de
un préstamo**— y lo haces razonar de las dos maneras clásicas: **hacia delante** (de los datos a la
conclusión) y **hacia atrás** (de la conclusión a los datos). Verás que llegan al mismo veredicto por
caminos distintos, que cuesta distinto número de comprobaciones, que la **resolución de conflictos** decide
qué regla manda cuando varias compiten, cómo propagar un **factor de certeza** (el puente con el capítulo 3)
y cómo el sistema **explica su razonamiento**, justo lo que una caja negra no sabe hacer.

### Qué vas a aprender
- Qué es una **base de conocimiento** (hechos + reglas) y un **motor de inferencia**.
- **Encadenamiento hacia delante** (dirigido por datos): el bucle emparejar–resolver–disparar hasta el punto fijo, con traza.
- **Encadenamiento hacia atrás** (dirigido por objetivos): la prueba recursiva de una meta, con su árbol de subobjetivos.
- **Resolución de conflictos**: orden, especificidad y recencia, y cómo cambian lo que se dispara.
- **Factores de certeza**: propagar la incertidumbre por la cadena de reglas (enlaza con el capítulo 3).
- El **módulo de explicación**: el «por qué» que distingue un sistema experto de una caja negra.

### Prerrequisitos
- Python básico (listas, diccionarios, funciones, recursión). Nada de aprendizaje automático.
- Haber leído el capítulo 4 del facsímil ayuda, pero el cuaderno es autocontenido.

### Cuánto cuesta
- Se ejecuta en segundos en la CPU de Colab. Solo usa `numpy` y `matplotlib` para una gráfica final.

> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

## 1. Conocimiento explícito: reglas y un motor que las encadena

Un **sistema experto** tiene dos piezas separadas a propósito:

1. La **base de conocimiento**: lo que sabemos del dominio, escrito como **hechos** (cosas que son ciertas
   ahora) y **reglas** *SI antecedentes ENTONCES consecuente*. La escribe un humano experto.
2. El **motor de inferencia**: un programa **genérico** que no sabe nada de préstamos ni de medicina;
   solo sabe **encadenar reglas**. El mismo motor sirve para cualquier dominio si cambias las reglas.

Esa separación es la idea potente: el conocimiento es **declarativo** (describes *qué* es verdad, no *cómo*
calcularlo) y **auditable** (puedes leer cada regla). Frente a una red neuronal, que guarda su «saber» en
millones de pesos ininteligibles, aquí el saber está en frases que cualquiera puede revisar y corregir.

Hay dos formas de encadenar, y son el corazón del capítulo:

- **Hacia delante** (*forward*, dirigido por datos): parte de los hechos conocidos y dispara reglas para
  derivar hechos nuevos, una y otra vez, hasta que no se puede derivar nada más. «Sé esto y esto, ¿qué se
  sigue?»
- **Hacia atrás** (*backward*, dirigido por objetivos): parte de una pregunta («¿aprobamos el préstamo?») y
  busca reglas que la respondan, generando subobjetivos que prueba recursivamente. «Quiero demostrar esto,
  ¿qué necesito?»

Construyamos primero la base de conocimiento del caso.

## 2. El caso: aprobar (o no) un préstamo personal

Una entidad evalúa solicitudes de préstamo con políticas escritas por su departamento de riesgos. Cada
política es una regla. Modelamos cada **hecho** como una cadena de texto (un símbolo que es cierto o no se
conoce) y cada **regla** como una estructura con nombre, lista de **antecedentes**, un **consecuente** y un
factor de certeza (lo usaremos más adelante).

Empezamos por una solicitante concreta, **Lucía**, de la que el formulario nos da estos hechos de partida.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Regla:
    nombre: str
    antecedentes: list      # lista de hechos que deben cumplirse
    consecuente: str        # hecho que se deriva si se cumplen
    cf: float = 1.0         # factor de certeza de la regla (cap. 3); 1.0 = certeza total
    texto: str = ""         # version legible, para la traza y la explicacion

# --- Reglas de la politica de riesgos (la "base de conocimiento") ---
BASE_REGLAS = [
    Regla("R1", ["empleo_fijo", "antiguedad_alta"], "ingresos_estables", 0.90,
          "SI empleo fijo Y antiguedad alta ENTONCES ingresos estables"),
    Regla("R2", ["sin_impagos"], "buen_historial", 1.00,
          "SI sin impagos ENTONCES buen historial crediticio"),
    Regla("R3", ["ratio_deuda_bajo", "ahorro_suficiente"], "capacidad_pago", 0.80,
          "SI ratio de deuda bajo Y ahorro suficiente ENTONCES capacidad de pago"),
    Regla("R4", ["ingresos_estables", "buen_historial"], "perfil_solvente", 0.90,
          "SI ingresos estables Y buen historial ENTONCES perfil solvente"),
    Regla("R5", ["perfil_solvente", "capacidad_pago"], "aprobar_prestamo", 0.95,
          "SI perfil solvente Y capacidad de pago ENTONCES aprobar prestamo"),
    # Reglas para el lado negativo (no se dispararan con Lucia, pero el motor las tiene):
    Regla("R6", ["impagos_recientes"], "riesgo_alto", 0.95,
          "SI impagos recientes ENTONCES riesgo alto"),
    Regla("R7", ["riesgo_alto"], "denegar_prestamo", 0.90,
          "SI riesgo alto ENTONCES denegar prestamo"),
    Regla("R8", ["empleo_temporal", "ratio_deuda_alto"], "denegar_prestamo", 0.85,
          "SI empleo temporal Y ratio de deuda alto ENTONCES denegar prestamo"),
]

# --- Hechos de partida de Lucia (lo que dice su formulario) ---
HECHOS_LUCIA = {"empleo_fijo", "antiguedad_alta", "sin_impagos",
                "ratio_deuda_bajo", "ahorro_suficiente"}

print(f"La base de conocimiento tiene {len(BASE_REGLAS)} reglas:")
for r in BASE_REGLAS:
    print(f"  {r.nombre}: {r.texto}  (cf={r.cf})")
print(f"\nHechos de partida de Lucia ({len(HECHOS_LUCIA)}): {sorted(HECHOS_LUCIA)}")
print(f"Pregunta del negocio: ¿debemos concluir 'aprobar_prestamo'?")

Fíjate en la estructura encadenable: R1 y R2 derivan hechos intermedios (`ingresos_estables`,
`buen_historial`) que R4 necesita; R4 y R3 alimentan a R5, que es la que decide. Ninguna regla habla de la
decisión final directamente desde los datos del formulario: hay que **encadenar** varias. Las reglas R6–R8
existen para el lado negativo; con los hechos de Lucía no se activarán, pero el motor las considerará
(eso cuenta, y lo mediremos).

## 3. Encadenamiento hacia delante (dirigido por datos)

El algoritmo es un bucle de tres pasos que se repite hasta el **punto fijo** (cuando una pasada completa no
deriva ningún hecho nuevo):

1. **Emparejar** (*match*): ¿qué reglas tienen todos sus antecedentes entre los hechos conocidos y aún no
   han aportado su consecuente? Ese es el **conjunto conflicto**.
2. **Resolver conflictos**: si hay varias, elegir cuál disparar (de momento, en orden).
3. **Disparar** (*fire*): añadir el consecuente a los hechos y repetir.

Lo implementamos contando cada vez que **comprobamos los antecedentes de una regla** (esa es la unidad de
trabajo que luego compararemos con el encadenamiento hacia atrás), y dejando una **traza** legible.

In [ ]:
def encadenar_adelante(hechos_iniciales, reglas, verbose=True):
    """Forward chaining: deriva hechos hasta el punto fijo. Devuelve (hechos, disparos, comprobaciones)."""
    hechos = set(hechos_iniciales)
    disparos = []          # nombres de reglas disparadas, en orden
    comprobaciones = 0     # cuantas veces evaluamos los antecedentes de una regla
    pasada = 0
    while True:
        pasada += 1
        disparo_en_pasada = False
        if verbose:
            print(f"--- Pasada {pasada} ---")
        for r in reglas:
            comprobaciones += 1
            ya_conocido = r.consecuente in hechos
            cumple = all(a in hechos for a in r.antecedentes)
            if cumple and not ya_conocido:
                hechos.add(r.consecuente)
                disparos.append(r.nombre)
                disparo_en_pasada = True
                if verbose:
                    ant = " ∧ ".join(r.antecedentes)
                    print(f"  DISPARA {r.nombre}: [{ant}] derivan -> '{r.consecuente}'")
        if not disparo_en_pasada:
            if verbose:
                print(f"  (ninguna regla nueva dispara: punto fijo alcanzado)")
            break
    return hechos, disparos, comprobaciones

hechos_fin, disparos_fwd, comprob_fwd = encadenar_adelante(HECHOS_LUCIA, BASE_REGLAS)

print()
print(f"Reglas disparadas, en orden: {disparos_fwd}")
print(f"¿Se derivo 'aprobar_prestamo'? {'aprobar_prestamo' in hechos_fin}")
print(f"Comprobaciones de regla realizadas: {comprob_fwd}")
print(f"Hechos nuevos derivados: {sorted(hechos_fin - HECHOS_LUCIA)}")

Lee la traza de arriba abajo: en una sola pasada se encadenan R1, R2 y R3 (derivan los hechos intermedios),
y como el bucle sigue recorriendo reglas en la misma pasada, R4 y R5 ya encuentran lo que necesitan y
disparan también. La segunda pasada no añade nada: **punto fijo**. El motor ha partido de cinco datos del
formulario y ha llegado a la decisión sin que nadie le dijera *cómo*; solo encadenando reglas.

## 4. Resolución de conflictos: ¿qué regla mando cuando varias compiten?

En cada pasada puede haber **varias reglas listas para disparar** a la vez: ese es el **conjunto conflicto**.
El orden en que las disparas puede cambiar el resultado. Las estrategias clásicas son:

- **Orden** (textual): dispara la primera según el orden en que se escribieron las reglas. Simple y
  predecible.
- **Especificidad**: dispara la regla **más específica** (la que tiene más antecedentes), porque encaja con
  una situación más concreta y suele ser la más informada.
- **Recencia**: dispara la regla que usa el **hecho derivado más recientemente**, para seguir la línea de
  razonamiento más «caliente».

Lo vemos con una decisión añadida: ofrecer a Lucía un **tipo de interés preferente** o el **estándar**. Dos
reglas compiten por concluir el tipo, y según la estrategia gana una u otra.

In [ ]:
# Dos reglas en conflicto: ambas se pueden disparar con el perfil de Lucia.
Ra = Regla("Ra", ["perfil_solvente"], "tipo_estandar", 0.9,
           "SI perfil solvente ENTONCES tipo de interes estandar")
Rb = Regla("Rb", ["perfil_solvente", "ahorro_suficiente"], "tipo_preferente", 0.9,
           "SI perfil solvente Y ahorro suficiente ENTONCES tipo de interes preferente")

conjunto_conflicto = [Ra, Rb]   # ambas tienen sus antecedentes satisfechos
print("Conjunto conflicto (ambas listas para disparar):")
for r in conjunto_conflicto:
    print(f"  {r.nombre}: {r.texto}  ({len(r.antecedentes)} antecedentes)")

def resolver(conjunto, estrategia):
    if estrategia == "orden":
        return conjunto[0]
    if estrategia == "especificidad":
        return max(conjunto, key=lambda r: len(r.antecedentes))
    raise ValueError(estrategia)

print()
elegida_orden = resolver(conjunto_conflicto, "orden")
elegida_espec = resolver(conjunto_conflicto, "especificidad")
print(f"Estrategia ORDEN          -> dispara {elegida_orden.nombre} -> '{elegida_orden.consecuente}'")
print(f"Estrategia ESPECIFICIDAD  -> dispara {elegida_espec.nombre} -> '{elegida_espec.consecuente}'")
print()
print("Misma situacion, distinto resultado: la estrategia NO es un detalle, es politica de negocio.")

Con **orden** gana Ra y Lucía recibe el tipo estándar; con **especificidad** gana Rb (tiene un antecedente
más) y recibe el tipo preferente. La misma base de reglas y los mismos hechos producen **decisiones
distintas** según cómo resuelvas el conflicto. Por eso un sistema experto serio documenta su estrategia: es
parte de la política, no un capricho técnico. La **recencia** seguiría la cadena más reciente; en motores
reales (como Rete) se combinan varios criterios con prioridades.

## 5. Encadenamiento hacia atrás (dirigido por objetivos)

Ahora invertimos la pregunta. En vez de «¿qué se sigue de los datos?», preguntamos «¿podemos **demostrar**
`aprobar_prestamo`?». El algoritmo es una **prueba recursiva**:

- Si el objetivo ya es un hecho conocido, está demostrado.
- Si no, busca reglas cuyo **consecuente** sea el objetivo. Para cada una, intenta demostrar **todos** sus
  antecedentes como **subobjetivos** (recursión). Si lo consigue con alguna regla, el objetivo queda
  demostrado.

Esto genera un **árbol de objetivos**: el objetivo en la raíz, sus subobjetivos como ramas, hasta llegar a
hechos del formulario en las hojas. Lo trazamos con sangría para ver el árbol, y contamos las mismas
**comprobaciones de regla** que antes, para comparar.

In [ ]:
def encadenar_atras(objetivo, hechos, reglas, contador, nivel=0, verbose=True):
    """Backward chaining: intenta demostrar 'objetivo'. 'contador' es una lista [n] mutable."""
    sangria = "  " * nivel
    if objetivo in hechos:
        if verbose:
            print(f"{sangria}- '{objetivo}': es un HECHO del formulario. [demostrado]")
        return True
    # Buscar reglas que concluyan el objetivo
    candidatas = [r for r in reglas if r.consecuente == objetivo]
    if not candidatas:
        if verbose:
            print(f"{sangria}- '{objetivo}': no es hecho ni hay regla que lo concluya. [falla]")
        return False
    for r in candidatas:
        contador[0] += 1   # comprobamos los antecedentes de esta regla
        if verbose:
            print(f"{sangria}- meta '{objetivo}': probar con {r.nombre} -> necesita {r.antecedentes}")
        if all(encadenar_atras(a, hechos, reglas, contador, nivel + 1, verbose)
               for a in r.antecedentes):
            if verbose:
                print(f"{sangria}  => {r.nombre} se cumple, '{objetivo}' queda DEMOSTRADO.")
            return True
    return False

contador_bwd = [0]
print("Arbol de objetivos para demostrar 'aprobar_prestamo':\n")
ok = encadenar_atras("aprobar_prestamo", HECHOS_LUCIA, BASE_REGLAS, contador_bwd)
print(f"\n¿'aprobar_prestamo' demostrado? {ok}")
print(f"Comprobaciones de regla realizadas: {contador_bwd[0]}")

El árbol se lee de la raíz a las hojas: para `aprobar_prestamo` el motor usa R5, que abre dos subobjetivos
(`perfil_solvente` y `capacidad_pago`); el primero abre a su vez R4 con `ingresos_estables` y
`buen_historial`, y así hasta tocar hechos del formulario (las hojas marcadas «es un HECHO»). El motor
**solo ha mirado las reglas que llevan a la meta**: nunca tocó R6, R7 ni R8, porque no concluyen nada que
necesitara. Esa es la ventaja del razonamiento dirigido por objetivos: no deriva todo, solo lo que hace
falta para la pregunta.

## 6. Mismo destino, distinto camino

Las dos estrategias llegan a la **misma conclusión** —aprobar el préstamo— pero recorriendo la base de
reglas de forma muy distinta. Hacia delante deriva **todo lo derivable** (aunque no lo hayamos pedido) y
revisa todas las reglas en cada pasada; hacia atrás se centra en la meta y **solo evalúa las reglas que la
sostienen**. Pongámoslo en números.

In [ ]:
print(f"{'Estrategia':<28}{'¿Aprueba?':<12}{'Comprobaciones de regla':<26}")
print("-" * 66)
print(f"{'Hacia delante (datos)':<28}{str('aprobar_prestamo' in hechos_fin):<12}{comprob_fwd:<26}")
print(f"{'Hacia atras (objetivos)':<28}{str(ok):<12}{contador_bwd[0]:<26}")
print()
print(f"Hacia delante derivo de mas {len(hechos_fin - HECHOS_LUCIA)} hechos intermedios y reviso todas las")
print(f"reglas hasta el punto fijo ({comprob_fwd} comprobaciones). Hacia atras solo siguio la cadena")
print(f"de la meta ({contador_bwd[0]} comprobaciones), sin tocar las reglas del lado negativo (R6-R8).")
print()
print("Regla practica: si tienes muchos datos y quieres ver TODO lo que se sigue, usa hacia delante.")
print("Si tienes UNA pregunta concreta y muchas reglas, hacia atras suele evaluar bastantes menos.")

## 7. Un toque de incertidumbre: factores de certeza

Hasta ahora un hecho era cierto o desconocido, todo o nada. Pero el conocimiento real tiene matices: «un
empleo fijo *sugiere* ingresos estables, no lo *garantiza*». Los **factores de certeza** (CF), del sistema
MYCIN que viste en el capítulo 3, ponen número a esa confianza, entre 0 y 1. Cada hecho tiene un CF y cada
regla tiene el suyo. La propagación por una regla es:

$$CF(\text{consecuente}) = CF(\text{regla}) \times \min_i CF(\text{antecedente}_i)$$

Es decir: una conclusión es como mucho tan fuerte como su antecedente **más débil** (el `min`, la cadena se
rompe por el eslabón más flojo), rebajada además por la confianza de la regla. Propaguémoslo por toda la
cadena de Lucía.

In [ ]:
# CF de los hechos de partida (confianza del formulario en cada dato)
CF_HECHOS = {
    "empleo_fijo": 0.90, "antiguedad_alta": 0.80, "sin_impagos": 1.00,
    "ratio_deuda_bajo": 0.90, "ahorro_suficiente": 0.70,
}

def encadenar_adelante_cf(cf_hechos, reglas):
    """Forward chaining propagando factores de certeza. Devuelve dict hecho -> CF."""
    cf = dict(cf_hechos)
    while True:
        cambio = False
        for r in reglas:
            if all(a in cf for a in r.antecedentes):
                cf_nuevo = round(r.cf * min(cf[a] for a in r.antecedentes), 4)
                if r.consecuente not in cf or cf_nuevo > cf[r.consecuente]:
                    if cf.get(r.consecuente) != cf_nuevo:
                        cambio = True
                    cf[r.consecuente] = cf_nuevo
                    eslabon = min(r.antecedentes, key=lambda a: cf[a])
                    print(f"{r.nombre}: CF('{r.consecuente}') = {r.cf} x min(...)={cf[min(r.antecedentes, key=lambda a: cf[a])]:.2f} "
                          f"(eslabon mas debil: '{eslabon}') = {cf_nuevo:.4f}")
        if not cambio:
            break
    return cf

print("Propagacion de la certeza por la cadena de Lucia:\n")
cf_final = encadenar_adelante_cf(CF_HECHOS, BASE_REGLAS)
cf_aprobar = cf_final["aprobar_prestamo"]
print(f"\nCF final de 'aprobar_prestamo' = {cf_aprobar:.4f}  (~{round(cf_aprobar*100)}% de certeza)")
print(f"Lectura: aprobamos, pero con una certeza del {round(cf_aprobar*100)}%, no del 100%.")

La conclusión no es un «sí» rotundo sino un «sí con matices». Observa que el CF final no depende del dato
más fuerte sino del recorrido por los **eslabones débiles**: el ahorro (CF 0.70) y las confianzas de las
reglas R3 y R5 arrastran la certeza hacia abajo. Esto es honesto: el sistema aprueba, pero te dice **cuánta
confianza** merece esa aprobación, algo que un umbral binario esconde. Cuando dos reglas distintas apoyan el
**mismo** hecho, MYCIN las combina con $CF_{comb}(a,b)=a+b(1-a)$ (la evidencia se acumula); aquí cada hecho
lo deriva una sola regla, así que no hace falta combinar.

## 8. El módulo de explicación: el «por qué»

Aquí está la diferencia decisiva con una caja negra. Como el motor ha encadenado reglas explícitas, puede
**reconstruir el razonamiento**: para cualquier conclusión, sabe qué regla la produjo y de qué hechos. Un
modelo opaco te da la respuesta; un sistema experto te da la respuesta **y la cadena de reglas que la
justifica**, en lenguaje que un humano puede auditar (y rebatir).

In [ ]:
def por_que(objetivo, hechos, reglas, nivel=0):
    """Explica POR QUE se concluyo 'objetivo', reconstruyendo la cadena de reglas."""
    sangria = "   " * nivel
    if objetivo in hechos:
        print(f"{sangria}* '{objetivo}' es un dato de partida (consta en el formulario).")
        return
    for r in reglas:
        if r.consecuente == objetivo and all(_demostrable(a, hechos, reglas) for a in r.antecedentes):
            ant = " y ".join(f"'{a}'" for a in r.antecedentes)
            print(f"{sangria}* '{objetivo}' porque {r.nombre}: {r.texto.lower()}.")
            print(f"{sangria}  Se cumple gracias a {ant}:")
            for a in r.antecedentes:
                por_que(a, hechos, reglas, nivel + 1)
            return

def _demostrable(objetivo, hechos, reglas):
    if objetivo in hechos:
        return True
    return any(r.consecuente == objetivo and all(_demostrable(a, hechos, reglas) for a in r.antecedentes)
               for r in reglas)

print("Pregunta del cliente: «¿Por que me habeis aprobado el prestamo?»\n")
por_que("aprobar_prestamo", HECHOS_LUCIA, BASE_REGLAS)

Eso es **trazabilidad**: cada paso de la decisión es una frase revisable. Si Lucía reclama, el banco puede
mostrarle exactamente qué reglas se aplicaron; si una política es injusta o ilegal, un auditor la localiza y
la corrige sin tocar el motor. Una red neuronal que rechaza un préstamo no puede ofrecer esto de forma
nativa: por eso, en dominios regulados (banca, sanidad, sector público), la **explicabilidad** de los
sistemas basados en reglas sigue siendo un valor, y por eso hoy se intenta dotar a los modelos opacos de
explicaciones aproximadas.

## 9. Visualización: el coste de cada estrategia

Una imagen para fijar la comparación del apartado 6: cuántas comprobaciones de regla hace cada motor para
llegar a la misma conclusión sobre el caso de Lucía.

In [ ]:
import matplotlib.pyplot as plt

estrategias = ["Hacia delante\n(datos)", "Hacia atras\n(objetivos)"]
costes = [comprob_fwd, contador_bwd[0]]

fig, ax = plt.subplots(figsize=(5, 3.4))
barras = ax.bar(estrategias, costes, color=["#4477aa", "#aa6644"])
ax.set_ylabel("Comprobaciones de regla")
ax.set_title("Mismo veredicto (aprobar), distinto coste")
for b, c in zip(barras, costes):
    ax.text(b.get_x() + b.get_width()/2, c + 0.2, str(c), ha="center", va="bottom", fontweight="bold")
ax.set_ylim(0, max(costes) + 3)
plt.tight_layout()
plt.show()

print(f"Hacia delante: {comprob_fwd} comprobaciones | Hacia atras: {contador_bwd[0]} comprobaciones.")

## 10. Pruébalo tú

El motor es genérico: cambia los **hechos** y la conclusión cambia sola, sin tocar ni una línea del bucle.
Probemos con **Mario**, otra solicitante con un perfil de riesgo. Mira cómo el mismo encadenamiento hacia
delante dispara ahora las reglas del lado negativo y concluye lo contrario.

In [ ]:
# Mario: empleo temporal, ratio de deuda alto e impagos recientes.
HECHOS_MARIO = {"empleo_temporal", "ratio_deuda_alto", "impagos_recientes"}

print("Caso de Mario:\n")
hechos_m, disparos_m, comprob_m = encadenar_adelante(HECHOS_MARIO, BASE_REGLAS)
print()
print(f"Reglas disparadas: {disparos_m}")
print(f"¿Aprobar?  {'aprobar_prestamo' in hechos_m}")
print(f"¿Denegar?  {'denegar_prestamo' in hechos_m}")
print()
print("El mismo motor, con datos distintos, recorre la OTRA parte de la base de reglas (R6, R7, R8)")
print("y concluye denegar. El conocimiento esta en las reglas; el motor solo las encadena.")
print()
print("Ideas para experimentar tu mismo/a:")
print(" - Quita 'ahorro_suficiente' de Lucia: ¿se sigue aprobando? (rompe R3 -> R5)")
print(" - Da a Mario tambien 'empleo_fijo' y 'antiguedad_alta': ¿que reglas compiten ahora?")
print(" - Baja el cf de R5 a 0.5 y observa como cae el CF final de la aprobacion.")

## 11. Errores comunes

- **Confundir el motor con el conocimiento.** El bucle de inferencia es genérico y no cambia entre dominios;
  lo que cambia son las reglas. Si el sistema decide mal, casi siempre es la base de conocimiento la que
  hay que corregir, no el motor.
- **Olvidar la resolución de conflictos.** Cuando varias reglas pueden disparar, el resultado depende de la
  estrategia. Dejarla implícita («el orden en que las escribí») produce decisiones frágiles y difíciles de
  auditar.
- **Bucles infinitos sin punto fijo.** Si una regla puede volver a derivar un hecho ya conocido, el bucle no
  termina. Por eso solo disparamos reglas cuyo consecuente **aún no** está entre los hechos.
- **Tratar el CF como una probabilidad.** Los factores de certeza son una heurística de confianza, no
  probabilidades reales: no suman 1 ni cumplen las leyes de Bayes. Para incertidumbre rigurosa, las redes
  bayesianas del capítulo 2 son la herramienta adecuada.
- **Creer que más reglas siempre es mejor.** Bases de miles de reglas se vuelven contradictorias y opacas;
  la potencia del enfoque viene de reglas **pocas, claras y auditables**.

## 12. Qué te llevas

- Un **sistema experto** separa el **conocimiento** (hechos + reglas *SI… ENTONCES…*, escritos por humanos)
  del **motor de inferencia** (un programa genérico que solo encadena reglas).
- El **encadenamiento hacia delante** parte de los datos y deriva todo lo posible hasta el punto fijo; el
  **hacia atrás** parte de una meta y solo explora las reglas que la sostienen. Llegan a la **misma
  conclusión** por caminos distintos y con **distinto coste**.
- La **resolución de conflictos** (orden, especificidad, recencia) decide qué regla manda cuando varias
  compiten, y es **política**, no un detalle.
- Los **factores de certeza** propagan la confianza por la cadena: la conclusión es tan fuerte como su
  eslabón más débil. El puente con el capítulo 3.
- El **módulo de explicación** reconstruye el «por qué» regla a regla. Esa **trazabilidad** es lo que una
  caja negra no ofrece, y la razón por la que las reglas siguen vivas en dominios regulados.

**Para seguir:** el capítulo 5 recapitula lógica, conocimiento e incertidumbre; los sistemas expertos son el
ancestro directo de los modernos motores de reglas de negocio y del razonamiento simbólico que hoy se
combina con los modelos neuronales (la llamada IA *neurosimbólica*).

---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*